# Random Forest and XGBoost Models Script

## This script accomplishes the following:

In this script, we will run 2 models, to predict the number of conflicts in each ZIP code, in each month.
The predictions are generated in a traditional ML way. We split the data into train and test sets, train the random forest on the former and  make the predicictions on the later.
Hence, we mask data that we "already saw", and then evaluate the models performance by taking the deviation from our predictions to the actual data.

In this script, we **predict**, we don't **forecast**, as the data is present and has already taken place, hence giving us a way to actually evaluate the model.

## Find the following steps below:

1. **Random Forest Model**

- Define features and label (number of conflicts, for each ZIP in PA, for each month)
- Perform 5-fold time-series-cross-validation
- Split the data 70/30 train and test
- Normalize the sets
- Define the rf regressor and fit
- Perform feature importance analysis
- Run a Grid Search to find the optimal parameter values
- Re-fit the model
- Make predictions
- Evaluation of Predictions

2. **XGBoost Model**

- Define the xgb regressor and fit
- Perform feature importance analysis
- Run a Grid Search to find the optimal parameter values
- Re-fit the model
- Make predictions
- Evaluation of Predictions

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

In [17]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

### Load the data

In [18]:
# Load the data

data = pd.read_csv(os.path.join(project_root_dir, 'Dataset/acs_conflict_combined_cleaned.csv'))

#data = pd.read_csv(project_root_dir, 'acs_conflict_combined_cleaned.csv')

## 1. Random Forest

### Features and labels, cross-validation, split and normalization

In [19]:
train = data[data['Month'] <= 9]

test = data[data['Month'] > 9]

In [20]:
X_train = train.drop(['event_count', 'ZCTA', 'Month'], axis=1)
Y_train = train['event_count']


X_test = test.drop(['event_count', 'ZCTA', 'Month'], axis=1)
Y_test = test['event_count']

In [21]:
'''
# Define features and label

X = data.drop(['event_count', 'ZCTA', 'Month'], axis=1)
Y = data['event_count']
'''

"\n# Define features and label\n\nX = data.drop(['event_count', 'ZCTA', 'Month'], axis=1)\nY = data['event_count']\n"

In [22]:
# Time Series Cross Validation, 5 folds.

tscv = TimeSeriesSplit(n_splits=5)

In [23]:
'''
# Split the dataset into training and test sets
# 70/30 split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=1)
'''

'\n# Split the dataset into training and test sets\n# 70/30 split\n\nX_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=1)\n'

In [24]:
# Normalize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Define the regressor and run

In [25]:
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=100)

In [26]:
rf_regressor.fit(X_train_scaled, Y_train)

RandomForestRegressor(random_state=100)

### Feature importance

In [ ]:
# Which are the most important features?

feature_importances = rf_regressor.feature_importances_
sorted_indices = np.argsort(feature_importances)[::-1]

In [ ]:
'''
# Create the plot
plt.figure(figsize=(10, 8))
plt.title('Feature Importances', fontsize=20)
plt.bar(range(len(feature_importances)), feature_importances[sorted_indices], align='center', alpha=0.5)
plt.xticks(range(len(feature_importances)), feature_names[sorted_indices], rotation=90)
plt.ylabel('Relative Importance', fontsize=15)
plt.xlabel('Feature', fontsize=15)
plt.tight_layout()  # Adjusts the plot to ensure everything fits without overlapping
plt.show()
'''

### Run Grid Search to find the best parameters

In [27]:
# Grid Search for best parameters

# Adjust number of trees in forest, max depth of trees, and minimum samples per leaf

# n_estimators: Number of trees in the forest, to take into account for the prediction
# max_depth: Maximum depth of the tree. From root to leaf, basically controls complexity of the tree and can lead/prevent overfitting. The more splits, the more complex the tree and the more likely to fit noise.
# min_samples_split: Minimum number of samples required to split an internal node


# Prevents overfitting and improves the generalization of the model

param_grid = {
    'n_estimators': [10, 20, 40],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 7]
}

In [29]:
cv_rf = GridSearchCV(estimator=rf_regressor, param_grid=param_grid, cv=5, n_jobs=-1)
cv_rf.fit(X_train_scaled, Y_train)

## Run the predictions

### Without Best Parameters

In [46]:
Y_pred = rf_regressor.predict(X_test_scaled)

predictions = pd.DataFrame({'Actual': Y_test, 'Predicted': Y_pred})
print(Y_pred)

[0.00622858 0.00670006 0.00716701 ... 0.4348492  0.00636438 0.00622858]


In [47]:
mse = mean_squared_error(Y_test, Y_pred)

print('Mean Squared Error:', mse)

Mean Squared Error: 0.07406755829726698


### Including the Best Parameters

In [50]:
Y_pred_gs = cv_rf.predict(X_test_scaled)

predictions_gs = pd.DataFrame({'Actual': Y_test, 'Predicted': Y_pred_gs})
print(Y_pred_gs)

[0.00574218 0.00571976 0.00613604 ... 0.27918392 0.00569096 0.00574218]


In [49]:
mse_gs = mean_squared_error(Y_test, Y_pred_gs)

print('Mean Squared Error:', mse_gs)

Mean Squared Error: 0.0781439381243572


### Show predictions and model evaluations

In [28]:
ZIP_results = pd.merge(predictions, data[['ZCTA', 'Month']], left_index=True, right_index=True)

In [29]:
print(ZIP_results)

       Actual  Predicted     ZCTA  Month
2232      0.0   0.005742  15364.0      1
13760     0.0   0.005720  17702.0      9
559       0.0   0.006136  15061.0      8
7310      0.0   0.005742  16238.0      3
352       0.0   0.005720  15042.0      5
...       ...        ...      ...    ...
13909     0.0   0.005742  17742.0      2
10454     0.0   0.005742  16922.0      3
20010     0.0   0.279184  19104.0      7
20584     0.0   0.005691  19301.0      5
14396     0.0   0.005742  17832.0      9

[6599 rows x 4 columns]


In [30]:
ZIP_results = ZIP_results[ZIP_results['Predicted'] >= 1]

In [31]:
mse = mean_squared_error(Y_test, Y_pred_gs)

print('Mean Squared Error:', mse)

Mean Squared Error: 0.0781439381243572


In [ ]:
# results_df = pd.DataFrame(columns=['Model', 'RMSLE', 'RMSE', 'MSE', 'MAE'])
